# Final attributes for INN 7708044880

Notebook collects agreed dashboard attributes for one INN and selected month.
Excluded metrics: avg_acq_pct, avg_irf_pct, relation_to_gk, terminal_ownership, and section-6 efficiency fields.

In [ ]:
import re
import pandas as pd
from decimal import Decimal, InvalidOperation
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
report_month = '2026-05-01'  # first day of month
target_inn = '7708044880'
aur_rate = 1926.0
mem_limit = '10g'
sa_type = 'SA'  # include only SA contracts

month_start = pd.to_datetime(report_month).strftime('%Y-%m-%d')
month_end = (pd.to_datetime(report_month) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

print('report_month =', report_month)
print('month_start =', month_start)
print('month_end =', month_end)
print('target_inn =', target_inn)
print('sa_type =', sa_type)
print('mem_limit =', mem_limit)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_pos_terminals')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_int')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_tune')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_fix')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_plan')
    imp.execute('invalidate metadata ods.scd1_z_depart')
    imp.execute('invalidate metadata ods.scd1_z_branch')
    imp.execute('invalidate metadata ods.scd1_z_cl_corp')
    imp.execute('invalidate metadata sandbox_ai.shestopalov_terminal_amortization_oneoff')
    imp.execute('invalidate metadata ocrm_ul.s_org_ext')
    imp.execute('invalidate metadata cdiul.ext_id_org')

print('Impala initialized')

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

def to_sql_in_list(values):
    out = []
    for v in values:
        s = str(v).replace("'", "''")
        out.append("'" + s + "'")
    return ', '.join(out)

## 1) Base perimeter: active agreements on first day of month

In [ ]:
sql_base = f"""
select
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.c_agr_number as string) as contract_number,
    cast(a.acq_class as string) as contract_type,
    cast(a.d_valid_from as timestamp) as d_valid_from,
    cast(a.d_valid_to as timestamp) as d_valid_to,
    cast(c.c_inn as string) as inn,
    cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{target_inn}'
  and upper(trim(cast(a.acq_class as string))) = '{sa_type}'
  and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    base_df = imp.fetch(sql_base)

display(base_df)
print('base_rows =', len(base_df))
print('contracts_with_agr_id =', int(base_df['agr_id'].notna().sum()) if len(base_df) else 0)
print('contracts_without_agr_id =', int(base_df['agr_id'].isna().sum()) if len(base_df) else 0)

## 2) R2 and organizational attributes

In [ ]:
if len(base_df) == 0:
    r2_df = pd.DataFrame(columns=['agr_id','contract_number','inn','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly'])
else:
    # tier1: по agr_id
    agr_ids = sorted(base_df['agr_id'].dropna().astype(str).unique().tolist())
    if len(agr_ids):
        in_list = to_sql_in_list(agr_ids)
        sql_r2_tier1 = f"""
        with r2 as (
          select cast(m.id as string) as agr_id, m.c_cl_org, m.c_depart, m.c_tariff_plan
          from ods.scd1_z_r2_ip_merchants m
          where cast(m.id as string) in ({in_list})
        ),
        fix as (
          select cast(m.id as string) as agr_id,
                 max(cast(tf.c_summa as double)) as commission_monthly
          from ods.scd1_z_r2_ip_merchants m
          left join ods.scd1_z_r2_tariff_tune tt on tt.c_tariff_plan = m.c_tariff_plan
          left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
          where cast(m.id as string) in ({in_list})
          group by cast(m.id as string)
        )
        select
          r2.agr_id,
          cast(null as string) as contract_number,
          cast(null as string) as inn,
          cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
          cast(dep.c_name as string) as vsp_name,
          cast(dep.c_num as string) as vsp_code,
          cast(br.c_shortlabel as string) as filial_rf,
          cast(tp.c_name as string) as tariff_name,
          cast(fix.commission_monthly as double) as commission_monthly
        from r2
        left join ods.scd1_z_cl_corp corp on corp.id = r2.c_cl_org
        left join ods.scd1_z_depart dep on dep.id = r2.c_depart
        left join ods.scd1_z_branch br on br.id = dep.c_filial
        left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan
        left join fix on fix.agr_id = r2.agr_id
        """
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            r2_tier1_df = imp.fetch(sql_r2_tier1)
    else:
        r2_tier1_df = pd.DataFrame(columns=['agr_id','contract_number','inn','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly'])

    # tier2: fallback по ИНН + номеру договора (когда agr_id отсутствует)
    base_fallback = base_df[base_df['agr_id'].isna() & base_df['contract_number'].notna()].copy()
    if len(base_fallback):
        cntr_vals = sorted(base_fallback['contract_number'].astype(str).unique().tolist())
        cntr_in = to_sql_in_list(cntr_vals)
        sql_r2_tier2 = f"""
        with b as (
          select
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(a.n_cmp_client as string) as n_cmp_client
          from ods_alpha.scd1_agreements a
          join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
          where cast(a.c_agr_number as string) in ({cntr_in})
            and regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{target_inn}'
            and upper(trim(cast(a.acq_class as string))) = '{sa_type}'
        )
        select
          cast(null as string) as agr_id,
          b.contract_number,
          b.inn,
          cast(null as string) as ogrn,
          cast(null as string) as vsp_name,
          cast(null as string) as vsp_code,
          cast(null as string) as filial_rf,
          cast(null as string) as tariff_name,
          cast(null as double) as commission_monthly
        from b
        group by b.contract_number, b.inn
        """
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            r2_tier2_df = imp.fetch(sql_r2_tier2)
    else:
        r2_tier2_df = pd.DataFrame(columns=['agr_id','contract_number','inn','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly'])

    r2_df = pd.concat([r2_tier1_df, r2_tier2_df], ignore_index=True)

display(r2_df)
print('r2_rows =', len(r2_df))

## 3) Company-level operational attributes (retl/term/mcc/amortization)

In [ ]:
if len(base_df) == 0:
    cmp_df = pd.DataFrame(columns=['n_cmp_client','retl_cnt','term_cnt','mcc_any','amortization'])
else:
    cmp_ids = sorted(base_df['n_cmp_client'].dropna().astype(str).unique().tolist())
    cmp_in = to_sql_in_list(cmp_ids) if len(cmp_ids) else "''"

    sql_cmp = f"""
    with m as (
      select cast(n_cmp as string) as n_cmp_client,
             cast(c_nmrc as string) as c_nmrc,
             cast(n_mcc as string) as n_mcc
      from ods_alpha.scd1_merchants
      where cast(n_cmp as string) in ({cmp_in})
        and c_nmrc is not null
    ),
    retl as (
      select n_cmp_client, count(distinct c_nmrc) as retl_cnt,
             max(n_mcc) as mcc_any
      from m
      group by n_cmp_client
    ),
    term as (
      select m.n_cmp_client,
             count(distinct t.c_nter) as term_cnt
      from m
      join ods_alpha.scd1_pos_terminals t
        on t.c_nmrc = m.c_nmrc and t.c_nter is not null
      group by m.n_cmp_client
    ),
    amort as (
      select m.n_cmp_client,
             sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
      from m
      join ods_alpha.scd1_pos_terminals t
        on t.c_nmrc = m.c_nmrc and t.c_nter is not null
      left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
        on cast(am.c_nter as string) = cast(t.c_nter as string)
      group by m.n_cmp_client
    )
    select coalesce(r.n_cmp_client, t.n_cmp_client, a.n_cmp_client) as n_cmp_client,
           r.retl_cnt,
           t.term_cnt,
           r.mcc_any,
           a.amortization
    from retl r
    full outer join term t on t.n_cmp_client = r.n_cmp_client
    full outer join amort a on a.n_cmp_client = coalesce(r.n_cmp_client, t.n_cmp_client)
    """

    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        cmp_df = imp.fetch(sql_cmp)

display(cmp_df)
print('cmp_rows =', len(cmp_df))

## 4) Transaction and commissions by agreement (month)

In [ ]:
if len(base_df) == 0:
    trx_df = pd.DataFrame(columns=['n_agr','trx_cnt','trx_sum','commission_from_ops','int_component'])
else:
    n_agrs = sorted(base_df['n_agr'].dropna().astype(str).unique().tolist())
    agr_in = to_sql_in_list(n_agrs) if len(n_agrs) else "''"

    sql_trx = f"""
    with trx_base as (
      select cast(t.n_trx as string) as n_trx,
             cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    ),
    ta as (
      select cast(n_trx as string) as n_trx,
             cast(n_agr as string) as n_agr,
             coalesce(cast(n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq
      where cast(n_agr as string) in ({agr_in})
    ),
    tj as (
      select ta.n_agr, ta.n_trx, tb.n_amt_src, ta.n_amt_tax
      from ta join trx_base tb on tb.n_trx = ta.n_trx
    )
    select
      tj.n_agr,
      count(distinct tj.n_trx) as trx_cnt,
      sum(tj.n_amt_src) as trx_sum,
      sum(tj.n_amt_tax) as commission_from_ops,
      sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as int_component
    from tj
    left join ods_alpha.scd1_trx_int i on cast(i.n_trx as string) = tj.n_trx
    group by tj.n_agr
    """

    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        trx_df = imp.fetch(sql_trx)

display(trx_df)
print('trx_rows =', len(trx_df))

## 5) OCRM mapping for SSP and CDI_ID

In [ ]:
sql_ocrm = f"""
with t as (
  select
    cast(row_id as string) as row_id,
    regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') as inn,
    case
      when upper(trim(cast(x_area_resp as string))) like 'ДММБ%' or upper(trim(cast(x_area_resp as string))) like 'ДМСБ (МИ%)' then 'ДМ'
      when upper(trim(cast(x_area_resp as string))) like 'ДМСБ (МА%)' then 'ДМСБ'
      when upper(trim(cast(x_area_resp as string))) like 'ДМСБ (СР%)' or upper(trim(cast(x_area_resp as string))) like 'ДСБ%' then 'ДМСБ'
      when upper(trim(cast(x_area_resp as string))) like 'ДКБ%' then 'ДКБ'
      when upper(trim(cast(x_area_resp as string))) like 'ДРПА%' then 'ДРПА'
      when lower(trim(cast(x_area_resp as string))) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(x_area_resp as string)), '') is null then null
      else trim(cast(x_area_resp as string))
    end as ssp_ocrm,
    created,
    row_number() over (
      partition by regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '')
      order by created desc, row_id desc
    ) as rn
  from ocrm_ul.s_org_ext
  where x_removed_flg = 'N'
    and x_duplicate_flg = 'N'
    and regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') = '{target_inn}'
),
m as (
  select distinct
    cast(party_id as string) as cdi_id,
    cast(cmo_ext_party_source_id as string) as row_id
  from cdiul.ext_id_org
  where cmo_ext_source_system like 'OCRM%'
)
select
  t.inn,
  t.row_id,
  m.cdi_id,
  t.ssp_ocrm
from t
left join m on m.row_id = t.row_id
where t.rn = 1
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    ocrm_df = imp.fetch(sql_ocrm)

display(ocrm_df)
print('ocrm_rows =', len(ocrm_df))

## 5) Final dataset and KPI calculations

In [ ]:
attrs_df = base_df.copy()

if len(r2_df):
    r2_tier1 = r2_df[r2_df['agr_id'].notna()].copy()
    if len(r2_tier1):
        attrs_df = attrs_df.merge(
            r2_tier1[['agr_id','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly']].drop_duplicates('agr_id'),
            on='agr_id',
            how='left'
        )

if len(r2_df):
    r2_tier2 = r2_df[r2_df['agr_id'].isna()].copy()
    if len(r2_tier2):
        attrs_df = attrs_df.merge(
            r2_tier2[['inn','contract_number','ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly']].drop_duplicates(['inn','contract_number']),
            on=['inn','contract_number'],
            how='left',
            suffixes=('', '_tier2')
        )
        for c in ['ogrn','vsp_name','vsp_code','filial_rf','tariff_name','commission_monthly']:
            c2 = f"{c}_tier2"
            if c2 in attrs_df.columns:
                attrs_df[c] = attrs_df[c].where(attrs_df[c].notna(), attrs_df[c2]) if c in attrs_df.columns else attrs_df[c2]
                attrs_df.drop(columns=[c2], inplace=True)
if len(cmp_df):
    attrs_df = attrs_df.merge(cmp_df, on='n_cmp_client', how='left')
if len(trx_df):
    attrs_df = attrs_df.merge(trx_df, on='n_agr', how='left')
if len(ocrm_df):
    ocrm_join_df = ocrm_df[['inn', 'row_id', 'cdi_id', 'ssp_ocrm']].drop_duplicates().rename(columns={'row_id': 'ocrm_row_id'})
    attrs_df = attrs_df.merge(ocrm_join_df, on='inn', how='left')

for c in ['retl_cnt','term_cnt','trx_cnt','trx_sum','commission_from_ops','commission_monthly','int_component','amortization']:
    if c in attrs_df.columns:
        attrs_df[c] = pd.to_numeric(attrs_df[c], errors='coerce').fillna(0)

attrs_df['key_tier'] = attrs_df['agr_id'].apply(lambda x: 'tier1_agr_id' if pd.notna(x) and str(x).strip() != '' else 'tier2_inn_contract')
attrs_df['dq_block_reason'] = attrs_df.apply(
    lambda r: None if (pd.notna(r.get('contract_number')) and str(r.get('contract_number')).strip() != '') else 'missing_contract_number',
    axis=1
)

attrs_df['commission_total'] = attrs_df.get('commission_from_ops', 0) + attrs_df.get('commission_monthly', 0)
attrs_df['aur'] = attrs_df.get('term_cnt', 0) * aur_rate
attrs_df['nod'] = attrs_df.get('commission_total', 0) - attrs_df.get('int_component', 0)
attrs_df['fin_result'] = attrs_df.get('nod', 0) - attrs_df.get('aur', 0) - attrs_df.get('amortization', 0)
attrs_df['report_month'] = report_month

final_columns = [
    'report_month',
    'inn', 'company_name', 'n_cmp_client', 'cdi_id', 'ocrm_row_id', 'ssp_ocrm',
    'contract_type', 'key_tier', 'dq_block_reason',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name', 'ogrn', 'mcc_any',
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'nod', 'fin_result'
]
final_columns = [c for c in final_columns if c in attrs_df.columns]
final_df = attrs_df[final_columns].copy()

display(final_df)
print('final_rows =', len(final_df))
print('tier_counts =')
display(final_df['key_tier'].value_counts(dropna=False).reset_index(name='rows').rename(columns={'index': 'key_tier'}))

In [ ]:
# quick QC
qc = pd.DataFrame([
    {'metric': 'rows', 'value': len(final_df)},
    {'metric': 'distinct_agr_id', 'value': final_df['agr_id'].nunique() if 'agr_id' in final_df.columns else None},
    {'metric': 'sum_trx', 'value': float(final_df['trx_sum'].sum()) if 'trx_sum' in final_df.columns else None},
    {'metric': 'sum_commission_total', 'value': float(final_df['commission_total'].sum()) if 'commission_total' in final_df.columns else None},
    {'metric': 'sum_fin_result', 'value': float(final_df['fin_result'].sum()) if 'fin_result' in final_df.columns else None},
])
display(qc)